In [ ]:

"""I will train a logistic-regression credit-risk model on 2/3 of the rows,
then scores EVERY row and writes back an Excel file with new columns:
    - probability_of_default   (0-1, or "Not available")
    - credit_score              (100-9990, or "Not available")
    - credit_grade               (A/B/C/D/E, or "Not available")
    - missing_fields             (which inputs, if any, were blank for that row)

MISSING-VALUE RULE:
    Three fields -- loan_amount_eur, loan_term_months, existing_debt_to_sales --
    are allowed to be blank. If ONLY fields from that list are missing for a
    merchant, the score is still calculated, using ONLY the variables that
    ARE available -- the missing ones are left out of the calculation
    entirely rather than imputed.

    If ANY other field is missing (or blank on top of the three above),
    there isn't enough reliable information to trust a score, so the
    merchant gets "Not available" instead of a number.

MONOTONIC CONSTRAINT ON BUSINESS AGE:
    Diagnostics on this dataset showed business_age_months has essentially
    no real relationship with default (correlation 0.016, p=0.39 -- not
    statistically significant), and the plain fitted model let a tiny,
    meaningless amount of noise push its coefficient to the WRONG side
    (older businesses scoring slightly worse), which contradicts sound
    underwriting logic. Rather than manufacturing an unsupported negative
    coefficient instead, the fit is constrained so business age's effect
    can only be zero or risk-REDUCING, never risk-increasing. On this data
    that constraint lands the coefficient at exactly 0.0 -- i.e. age ends
    up with no influence on the score, which is the most the data honestly
    supports. Every other coefficient is refit normally around that rule.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize


# ============================================================================
# CONFIGURATION - EDIT THESE PATHS
# ============================================================================
INPUT_FILE = r"C:\Users\Georgina\Downloads\synthetic_merchant_credit_data.csv"
OUTPUT_FILE = r"C:\Users\Georgina\Downloads\merchant_scores_final(German_Scale)(4).xlsx"

TARGET_COL = "default_flag"
ID_COL = "merchant_id"
LEAKAGE_COLS = ["event_observed", "duration_months", "active_days", "observed_days"]
CATEGORICAL_COLS = ["merchant_type"]

# Fields that are ALLOWED to be missing at scoring time -- if a merchant is
# missing ONLY fields from this list, they still get scored on whatever's left.
OPTIONAL_MISSING_COLS = ["loan_amount_eur", "loan_term_months", "existing_debt_to_sales"]

# Features whose effect on the score is NOT allowed to point the "wrong" way,
# regardless of what the raw fit says. Each entry: "max" means the coefficient
# must be <= 0 (the feature can only REDUCE risk, e.g. more business age should
# never increase risk); "min" means the coefficient must be >= 0 (the feature
# can only INCREASE risk, e.g. more unresolved critical issues should never
# reduce risk). Add more features here if the same issue shows up elsewhere.
MONOTONIC_CONSTRAINTS = {
    "business_age_months": "max",   # age may only help or do nothing -- never hurt
    "capture_quality": "max",       # higher capture quality may only help or do nothing -- never hurt
    "finalized_payments": "max",    # more finalized payments may only help or do nothing -- never hurt
}

TEST_SIZE = 1 / 3           # 2/3 train, 1/3 test
RANDOM_STATE = 42           # Set to None for random splits each time
REG_C = 1.0                 # L2 regularization strength (same convention as sklearn's C)
PDO = 40                    # Points to double the odds
BASE_SCORE = 600            # Score for average-risk merchant
# ============================================================================


def fit_constrained_logistic(X, y, C, monotonic_constraints, feature_names):
    """
    Fits a logistic regression by directly minimizing L2-regularized log-loss,
    the same objective sklearn's LogisticRegression(penalty='l2', C=C) uses --
    but allows bounding individual coefficients so their sign can be forced
    to match sound business logic when the data's own signal is too weak or
    noisy to trust on its own (see MONOTONIC_CONSTRAINTS above).

    Returns (intercept, coef_array).
    """
    n, p = X.shape

    def loss_and_grad(w):
        z = np.clip(X @ w[1:] + w[0], -30, 30)
        p_hat = 1 / (1 + np.exp(-z))
        eps = 1e-12
        loss = -np.sum(y * np.log(p_hat + eps) + (1 - y) * np.log(1 - p_hat + eps))
        loss += (1 / (2 * C)) * np.sum(w[1:] ** 2)
        grad_w = X.T @ (p_hat - y) + w[1:] / C
        grad_b = np.sum(p_hat - y)
        return loss, np.concatenate([[grad_b], grad_w])

    bounds = [(None, None)]  # intercept: unconstrained
    for name in feature_names:
        rule = monotonic_constraints.get(name)
        if rule == "max":
            bounds.append((None, 0))
        elif rule == "min":
            bounds.append((0, None))
        else:
            bounds.append((None, None))

    w0 = np.zeros(p + 1)
    result = minimize(loss_and_grad, w0, jac=True, method="L-BFGS-B", bounds=bounds)
    return result.x[0], result.x[1:]


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))


def build_credit_scores():
    """Run the full training + scoring pipeline."""

    print("=" * 80)
    print("MERCHANT CREDIT SCORING PIPELINE")
    print("=" * 80)

    # ---------- Stage 1: load & prepare features ----------
    print(f"\n[1/8] Loading data from:\n  {INPUT_FILE}")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print(f"ERROR: File not found at {INPUT_FILE}")
        print("Please check the path and try again.")
        return None

    print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

    if TARGET_COL not in df.columns:
        print(f"ERROR: Target column '{TARGET_COL}' not found.")
        print(f"Available columns: {df.columns.tolist()}")
        return None

    if df[TARGET_COL].nunique() < 2:
        print(f"ERROR: '{TARGET_COL}' only has one outcome value.")
        return None

    print(f"\n[2/8] Preparing features...")
    drop_cols = set([ID_COL, TARGET_COL]) | set(LEAKAGE_COLS)
    feature_cols = [c for c in df.columns if c not in drop_cols]

    print(f"Using {len(feature_cols)} features")
    print(f"Dropping {len(drop_cols)} columns (ID/leakage): {list(drop_cols)}")

    optional_cols_present = [c for c in OPTIONAL_MISSING_COLS if c in feature_cols]
    print(f"Fields allowed to be missing without penalty: {optional_cols_present}")
    print(f"Monotonic constraints applied: {MONOTONIC_CONSTRAINTS}")

    missing_mask = df[feature_cols].isna()
    missing_fields_per_row = missing_mask.apply(
        lambda r: [c for c in feature_cols if r[c]], axis=1
    )

    def classify_row(missing_list):
        if len(missing_list) == 0:
            return "complete"
        if set(missing_list).issubset(set(optional_cols_present)):
            return "partial_ok"
        return "not_available"

    row_status = missing_fields_per_row.apply(classify_row)
    print(f"Rows fully complete:            {(row_status == 'complete').sum()}")
    print(f"Rows missing only optional data: {(row_status == 'partial_ok').sum()}")
    print(f"Rows missing required data:      {(row_status == 'not_available').sum()} -> will be 'Not available'")

    cats_present = [c for c in CATEGORICAL_COLS if c in feature_cols]
    df_cat_filled = df[feature_cols].copy()
    for c in cats_present:
        df_cat_filled[c] = df_cat_filled[c].fillna("Missing")

    X = pd.get_dummies(df_cat_filled, columns=cats_present, drop_first=True)
    y = df[TARGET_COL].values

    print(f"After one-hot encoding: {X.shape[1]} features")
    print(f"Default rate: {y.mean():.1%}")

    # ---------- Stage 2: train/test split ----------
    print(f"\n[3/8] Splitting into train/test...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    print(f"Train: {len(X_train)} rows ({y_train.mean():.1%} default rate)")
    print(f"Test:  {len(X_test)} rows ({y_test.mean():.1%} default rate)")

    # ---------- Stage 3: impute TRAINING data only (median) ----------
    imputer = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

    # ---------- Stage 4: standardize features ----------
    print(f"\n[4/8] Standardizing features...")
    scaler = StandardScaler().fit(X_train_imp)
    X_train_s = scaler.transform(X_train_imp)
    X_test_s = scaler.transform(X_test_imp)
    print(f"Scaler fit on TRAIN only (to avoid leakage)")

    # ---------- Stage 5: fit logistic regression, with monotonic constraints ----------
    print(f"\n[5/8] Fitting logistic regression (monotonic constraints applied)...")
    intercept, coef_array = fit_constrained_logistic(
        X_train_s, y_train, REG_C, MONOTONIC_CONSTRAINTS, list(X.columns)
    )
    print(f"Model trained")
    for name, rule in MONOTONIC_CONSTRAINTS.items():
        i = list(X.columns).index(name)
        print(f"  {name}: coefficient = {coef_array[i]:+.6f}  (constraint: {rule})")

    # ---------- Stage 6: evaluate ----------
    print(f"\n[6/8] Evaluating model...")
    train_auc = roc_auc_score(y_train, sigmoid(X_train_s @ coef_array + intercept))
    test_auc = roc_auc_score(y_test, sigmoid(X_test_s @ coef_array + intercept))

    print(f"Train AUC: {train_auc:.3f}")
    print(f"Test AUC:  {test_auc:.3f}")

    if abs(train_auc - test_auc) > 0.05:
        print(f"  Gap > 0.05 suggests possible overfitting")
    else:
        print(f"  Small gap = good generalization")

    # ---------- Stage 7: score every row, honoring the missing-data rule ----------
    print(f"\n[7/8] Scoring all rows...")

    base_rate = y_train.mean()
    base_odds = (1 - base_rate) / base_rate
    factor = PDO / np.log(2)
    offset = BASE_SCORE - factor * np.log(base_odds)

    coef = pd.Series(coef_array, index=X.columns)
    means = pd.Series(scaler.mean_, index=X.columns)
    stds = pd.Series(scaler.scale_, index=X.columns)

    def grade(score):
        if score >= 776:
            return "Excellent"
        elif score >= 709:
            return "Good"
        elif score >= 642:
            return "Acceptable"
        elif score >= 100:
            return "Sufficient"
        return "E"

    probs, scores, grades, missing_field_strs = [], [], [], []

    for idx, status in row_status.items():
        missing_list = missing_fields_per_row.loc[idx]
        missing_field_strs.append(", ".join(missing_list) if missing_list else "")

        if status == "not_available":
            probs.append("Not available")
            scores.append("Not available")
            grades.append("Not available")
            continue

        skip_cols = set(missing_list) & set(optional_cols_present)
        use_cols = [c for c in X.columns if c not in skip_cols]

        row_x = X.loc[idx, use_cols]
        z = (row_x - means[use_cols]) / stds[use_cols]
        logit = intercept + float((z * coef[use_cols]).sum())

        p_default = 1 / (1 + np.exp(-logit))
        odds_good = (1 - p_default) / p_default
        score = offset + factor * np.log(odds_good)
        score = float(np.clip(score, 300, 850))

        probs.append(round(p_default, 4))
        scores.append(round(score))
        grades.append(grade(score))

    n_partial = (row_status == "partial_ok").sum()
    print(f"Scored {len(df) - (row_status == 'not_available').sum()} merchants "
          f"({n_partial} using a reduced set of variables)")
    print(f"Marked 'Not available': {(row_status == 'not_available').sum()} merchants")

    # ---------- Attach results & save ----------
    print(f"\n[8/8] Writing Excel file...")
    df_out = df.copy()
    df_out["probability_of_default"] = probs
    df_out["credit_score"] = scores
    df_out["credit_grade"] = grades
    df_out["missing_fields"] = missing_field_strs

    try:
        df_out.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')
        print(f"Successfully saved to:\n  {OUTPUT_FILE}")
    except Exception as e:
        print(f"Error saving Excel file: {e}")
        print("Make sure openpyxl is installed: pip install openpyxl")
        return None

    print(f"\n{'=' * 80}")
    print("SAMPLE RESULTS (first 10 merchants):")
    print(f"{'=' * 80}\n")
    sample = df_out[[ID_COL, "probability_of_default", "credit_score", "credit_grade", "missing_fields"]].head(10)
    print(sample.to_string(index=False))
    print(f"\n{'=' * 80}\n")

    print("CREDIT GRADE DISTRIBUTION:")
    print(df_out["credit_grade"].value_counts().sort_index())

    print(f"\nDONE! Excel file saved to Downloads folder.")

    return df_out


# ============================================================================
# RUN THE PIPELINE
# ============================================================================
if __name__ == "__main__":
    df_scored = build_credit_scores()